# 2 — Estágio B: voz do Pedro sobre a BASE-PT (HF puro) · Colab

Treina um LoRA da **sua voz** por cima da **BASE-PT** (vencedor da bateria de
língua, notebook 1b) — ou direto sobre o `csm-1b` se `BASE_PT_ADAPTER=None`.
**HF puro, sem Unsloth** (que quebra o forward do CSM — lição 11-15/jun):
`CsmForConditionalGeneration` + `peft`, `transformers==4.52.3`.

**Pré-requisito:** `dataset_v1` no Drive (notebook 0) + opcionalmente a BASE-PT
do notebook 1b. **GPU:** T4 ok (L4/A100 = folga). Geração SEMPRE com contexto
de referência (sem ele a voz varia).

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
GH_TOKEN = userdata.get('GH_TOKEN')
!git clone https://{GH_TOKEN}@github.com/pedrocormann/TTS-ptbr.git /content/TTS-ptbr 2>/dev/null || (cd /content/TTS-ptbr && git pull)
%cd /content/TTS-ptbr
!mkdir -p data && ln -sfn /content/drive/MyDrive/TTS-ptbr-data/dataset_v1 data/dataset_v1
!ls data/dataset_v1/

In [ ]:
%%capture
import os; os.environ['HF_HUB_ENABLE_HF_TRANSFER']='1'
# HF puro — SEM unsloth (quebra o CSM). transformers 4.52.3 = versão do checkpoint
# (5.x renomeia pesos → embed_audio_tokens MISSING). torchao velho do Colab → remover.
!pip install "transformers==4.52.3" peft accelerate "datasets>=3.4.1,<4.0.0" \
    soundfile jiwer librosa soxr bitsandbytes torchcodec faster-whisper==1.1.0 hf_transfer
!pip uninstall -y torchao

## 1. Dataset → formato CSM (conversa speaker 0, áudio 24 kHz, tags inline)

In [ ]:
import json, pathlib
from datasets import Dataset, Audio

ROOT = pathlib.Path('data/dataset_v1')
rows = [json.loads(l) for l in (ROOT/'orpheus_train.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]
# orpheus_*.jsonl já traz o texto com prefixo de estilo/sotaque (<animado|...>) — tags inline
raw_ds = Dataset.from_list([{'text': r['text'], 'audio': str(ROOT/r['audio'])} for r in rows])
raw_ds = raw_ds.cast_column('audio', Audio(sampling_rate=24000))
print(raw_ds)

# max_length de áudio calculado do NOSSO dataset (oficial usa 240001 = 10s fixos)
max_audio = max(len(ex['audio']['array']) for ex in raw_ds)
MAX_AUDIO = 288000 + 1   # 12s fixo (áudio cortado a 12s no preprocess)
MAX_TEXT = 384
print(f'max áudio: {max_audio/24000:.1f}s → max_length={MAX_AUDIO}')

In [ ]:
from transformers import AutoProcessor, CsmForConditionalGeneration
from peft import PeftModel
import torch

MODEL_ID = 'unsloth/csm-1b'              # mirror Apache ungated
# Estágio B: aponte p/ a BASE-PT (vencedor da bateria do notebook 1b) p/ a voz ser
# treinada SOBRE o português já aprendido. None = treina direto sobre o csm-1b base.
BASE_PT_ADAPTER = None  # ex: '/content/drive/MyDrive/TTS-ptbr-data/runs/battery_A3_tagarela/final'
BF16 = torch.cuda.is_bf16_supported()

processor = AutoProcessor.from_pretrained(MODEL_ID)
model, _info = CsmForConditionalGeneration.from_pretrained(
    MODEL_ID, output_loading_info=True)  # float32: o codec Mimi gera float32, então o modelo
    # NÃO pode ser bf16 (mismatch no merge). bf16=True no Trainer faz autocast.
assert not [k for k in _info.get('missing_keys', []) if 'embed_audio' in k], \
    '❌ pesos de ÁUDIO faltando — transformers incompatível com o checkpoint'
if BASE_PT_ADAPTER:                      # funde a língua (BASE-PT) no base → novo ponto de partida
    model = PeftModel.from_pretrained(model, BASE_PT_ADAPTER).merge_and_unload()
    print('✅ BASE-PT carregada e fundida:', BASE_PT_ADAPTER)
model = model.to('cuda'); model.train(); model.codec_model.eval()   # codec Mimi congelado

def preprocess(example):
    conversation = [{'role': '0', 'content': [        # speaker id 0 = Pedro
        {'type': 'text',  'text': example['text']},
        {'type': 'audio', 'path': example['audio']['array'][:288000]}]}]
    out = processor.apply_chat_template(
        conversation, tokenize=True, return_dict=True, output_labels=True,
        text_kwargs={'padding': 'max_length', 'max_length': MAX_TEXT,
                     'truncation': True, 'pad_to_multiple_of': 8, 'padding_side': 'right'},
        audio_kwargs={'sampling_rate': 24000, 'max_length': MAX_AUDIO, 'padding': 'max_length'},
        common_kwargs={'return_tensors': 'pt'})
    return {k: v[0] for k, v in out.items()}

processed_ds = raw_ds.map(preprocess, remove_columns=raw_ds.column_names)
print(processed_ds)

## 2. LoRA + treino (r=64 — voz é "hero"; comunidade TTS usa rank > texto)

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments, Trainer

cfg = LoraConfig(r=64, lora_alpha=64, lora_dropout=0.0, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
model = get_peft_model(model, cfg); model.print_trainable_parameters()

class CSMTrainer(Trainer):   # mantém o codec Mimi em eval (não treina)
    def training_step(self, model, inputs, *args, **kwargs):
        bm = model.get_base_model() if hasattr(model, 'get_base_model') else model
        if hasattr(bm, 'codec_model'): bm.codec_model.eval()
        return super().training_step(model, inputs, *args, **kwargs)

trainer = CSMTrainer(
    model=model, train_dataset=processed_ds,
    args=TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=8,   # efetivo 16
        warmup_ratio=0.03, num_train_epochs=3, learning_rate=2e-4,
        bf16=BF16, fp16=not BF16, logging_steps=5, optim='adamw_8bit',
        weight_decay=0.001, lr_scheduler_type='cosine', seed=3407,
        output_dir='outputs', report_to='none', save_steps=200, save_total_limit=2,
        remove_unused_columns=False))
trainer.train()

## 3. Geração com CONTEXTO de voz (obrigatório — sem contexto a voz varia)
Gera o benchmark congelado `eval/benchmark_ptbr.jsonl` com 1 utterance de referência.

In [ ]:
import soundfile as sf, json, pathlib

ref = raw_ds[0]                                   # utterance de referência (limpa, neutra)
bench = [json.loads(l) for l in open('eval/benchmark_ptbr.jsonl', encoding='utf-8') if l.strip()]
outdir = pathlib.Path('gen_csm'); outdir.mkdir(exist_ok=True)

model.eval()
for i, item in enumerate(bench):
    conversation = [
        {'role': '0', 'content': [{'type': 'text', 'text': ref['text']},
                                  {'type': 'audio', 'path': ref['audio']['array']}]},
        {'role': '0', 'content': [{'type': 'text', 'text': item['text']}]},
    ]
    inputs = processor.apply_chat_template(conversation, tokenize=True, return_dict=True)
    with torch.no_grad():
        audio = model.generate(**inputs.to('cuda'),
                               max_new_tokens = 375,    # ~30s de teto (125≈10s)
                               depth_decoder_do_sample=True, depth_decoder_temperature=0.9,
                               do_sample=True, temperature=0.9,
                               output_audio = True)
    wav = audio[0].to(torch.float32).cpu().numpy()
    sf.write(outdir / f"{item.get('id', i):0>3}.wav", wav, 24000)
    print(item['text'][:60])

## 4. Eval (gates F1 do REPLAN: spk-sim ≥ 0.70 · WER ≤ 1.2× real · escuta)

In [ ]:
!pip -q install faster-whisper==1.1.0
!python -m eval.wer_roundtrip --in-dir gen_csm --transcripts eval/benchmark_ptbr.jsonl --model medium --lang pt
# referência da voz real = clipes neutros do dataset
!mkdir -p ref_pedro && python -c "
import json, shutil, pathlib
rows=[json.loads(l) for l in open('data/dataset_v1/train.jsonl', encoding='utf-8') if l.strip()]
neutros=[r for r in rows if r.get('style')=='neutro'][:20]
[shutil.copy(pathlib.Path('data/dataset_v1')/r['audio'], 'ref_pedro/') for r in neutros]"
!python -m eval.speaker_sim --ref-dir ref_pedro --gen-dir gen_csm

## 5. Salvar (adapter + merge) no Drive — o "ouro" fica fora do git

In [ ]:
SAVE = '/content/drive/MyDrive/TTS-ptbr-data/checkpoints/csm_voz_pedro_v1'
model.save_pretrained(SAVE); processor.save_pretrained(SAVE)   # salva o adapter LoRA da voz
print('✅ adapter salvo em', SAVE)
# reload em sessão nova (HF puro):
#   from transformers import CsmForConditionalGeneration, AutoProcessor
#   from peft import PeftModel
#   base = CsmForConditionalGeneration.from_pretrained('unsloth/csm-1b', torch_dtype='auto').to('cuda')
#   model = PeftModel.from_pretrained(base, SAVE)          # ou .merge_and_unload() p/ servir
#   processor = AutoProcessor.from_pretrained(SAVE)
# p/ usar no app local (src/duplex): copie o adapter pro Mac e aponte o CSMAdapter